# Atelier Préparation de Données Tabulaires
## Bâtiments intelligents — capteurs IoT

**Contexte :** une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT.
Chaque capteur collecte des informations sur la température, l'humidité, la qualité de l'air (CO₂),
la consommation énergétique, l'occupation, le type de bâtiment, le mode de climatisation et l'état
du système.

**Objectif :** transformer le fichier brut `smart_building_raw.csv` en un jeu de données propre,
prêt à être utilisé par un algorithme de Machine Learning (prédiction de consommation énergétique
ou détection d'anomalies via la variable `alerte`).

---
# Partie 1 — Explorer les données

<mark style= "background-color: lightblue">**📋Question 1.1** </mark>

 Charger les données CSV

In [16]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_csv("../data/smart_building_raw.csv")

<mark style= "background-color: lightblue">**📋Question 1.2** </mark>

Afficher les preméres lignes du dataset


In [18]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


<mark style= "background-color: lightblue">**📋Question 1.3** </mark>

Afficher les derniéres lignes du dataset

In [19]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


<mark style= "background-color: lightblue">**📋Question 1.4** </mark>

Combien d'observations contient le dataset ?

In [20]:
num_observations = df.shape[0]
print(f"Nombre d'observations: {num_observations}")

Nombre d'observations: 507


<mark style= "background-color: lightblue">**📋Question 1.5** </mark>

Combien de variables possède le dataset ? 

In [21]:
num_variables = df.shape[1]
print(f"Nombre de variables: {num_variables}")

Nombre de variables: 14


<mark style= "background-color: lightblue">**📋Question 1.6** </mark>

Identifier les variables numériques ; 

Une variable numérique est une variable dont les valeurs sont des quantités mesurables (continues), à distinguer des identifiants qui, bien que numériques, ne représentent pas une quantité.

In [22]:
print("Types de variables:")
print(df.dtypes)

Types de variables:
id_mesure               int64
date                      str
batiment                  str
type_batiment             str
zone                      str
temperature           float64
humidite              float64
co2                   float64
occupation            float64
consommation_kwh      float64
mode_climatisation        str
etat_systeme              str
jour_semaine              str
alerte                    str
dtype: object


In [23]:
variables_num= df.select_dtypes(include=['number']).columns.tolist()
print ("Variables numériques:")
print(variables_num)

Variables numériques:
['id_mesure', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


<mark style= "background-color: lightblue">**📋Question 1.7** </mark>

Identifier les variables catégorielles

In [24]:
variables_cat = df.select_dtypes(include=['object', 'str','category']).columns.tolist()
print("Variables catégorielles:")
print(variables_cat)

Variables catégorielles:
['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


<mark style= "background-color: lightblue">**📋Question 1.8** </mark>

Identifier les dates



Le dataset contient une colonne **"date"** qui doit être exploitée comme telle, mais elle est
chargée par pandas comme une simple chaîne de caractères **(texte)** et non comme une vraie date.

On la convertit donc avec **pd.to_datetime()**, ce qui permettra ensuite de trier les mesures
chronologiquement ou d'en extraire des informations utiles (mois, heure, jour de la semaine...).

Le paramètre **errors="coerce"** indique que si une valeur ne peut pas être interprétée comme une
date valide, elle sera remplacée par **NaT** (Not a Time, l'équivalent de NaN pour les dates) plutôt
que de provoquer une erreur qui arrêterait tout le programme.

On vérifie enfin le type de la colonne après conversion pour confirmer qu'elle est bien passée
en datetime64.

In [25]:
dates = ["date"]
print("Colonne(s) de type date :", dates)

# Conversion au format datetime pour pouvoir l'exploiter (extraction mois/heure, tri, etc.)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(df["date"].dtype)

Colonne(s) de type date : ['date']
datetime64[us]


<mark style= "background-color: lightblue">**📋Question 1.9** </mark>

Identifier les identifiants 


In [26]:
identifiant = ["id_mesure"]
print("Colonne(s) identifiant :", identifiant)

Colonne(s) identifiant : ['id_mesure']


<mark style= "background-color: lightblue">**📋Question 1.10** </mark>

Déterminer les statistiques : moyenne, médiane, minimum, maximum, écart-type et quartiles. 

In [33]:
stat_numeriques = df[variables_num].describe().T
stat_numeriques["median"] = df[variables_num].median()
stat_numeriques[["min", "25%", "50%", "median", "75%", "max", "mean", "std"]]



,min,25%,50%,median,75%,max,mean,std
id_mesure,1001.0,1125.500,1252.00,1252.00,1376.500,1500.0,1251.114398,144.782769
temperature,-30.0,21.600,24.00,24.00,26.000,96.0,24.154141,7.418465
humidite,-12.0,49.275,57.55,57.55,65.750,160.0,57.864113,16.026336
co2,89.0,623.750,787.50,787.50,952.000,6000.0,844.150000,582.181386
occupation,-20.0,27.000,46.00,46.00,61.000,116.0,44.850299,24.949139
consommation_kwh,-100.0,136.875,169.80,169.80,202.975,336.2,169.069323,53.164294


<mark style= "background-color: lightblue">**📋Question 1.11** </mark>

Y a-t-il des variables potentiellement problématiques ?

En observant les `min`/`max` ci-dessus :
- **temperature** : min à -30°C, max à 96°C → impossible physiquement.
- **humidite** : min négatif, max à 160% → une humidité relative est comprise entre 0 et 100%.
- **occupation** : valeurs négatives → un nombre de personnes ne peut pas être négatif.
- **consommation_kwh** : valeurs négatives → une consommation ne peut pas être négative.
- **co2** : max à 6000 ppm, élevé mais pas impossible (valeur extrême à surveiller).


<mark style= "background-color: lightblue">**📋Question 1.12-— Données incohérentes** </mark>

 **12.a** : rechercher des valeurs telles que humidité < 0 ;

humidite_neg = df["humidite"] < 0
print(f"{humidite_neg.sum()} valeurs d'humidité négatives :")
df.loc[humidite_neg, ["id_mesure","humidite"]]
